# AC power flow on the SimPT-Power HV core
Solve the 60–400 kV core with pandapower, reproduce the national peak snapshot, and run a simple N-1 check.
Pinned version: the release was computed with **pandapower 3.5.2**.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("pandapower") is None:  # on Kaggle: enable Internet in the notebook settings
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandapower==3.5.2"], check=True)
import pandapower as pp
print("pandapower", pp.__version__)

In [ ]:
import os, json
from pathlib import Path
def find_data():
    """Locate the SimPT-Power open bundle: $SIMPT_DATA, a Kaggle input dataset, or the local build folder."""
    cands = [os.environ.get("SIMPT_DATA", "")]
    cands += [str(p.parent) for p in Path("/kaggle/input").rglob("bundle_manifest.json")] if Path("/kaggle/input").exists() else []
    cands += ["../output/simpt_power_release/open_bundle_r2", "output/simpt_power_release/open_bundle_r2"]
    for c in cands:
        if c and (Path(c) / "bundle_manifest.json").exists():
            return Path(c)
    raise FileNotFoundError("SimPT-Power bundle not found; set SIMPT_DATA")
DATA = find_data()
manifest = json.loads((DATA / "bundle_manifest.json").read_text())
print(DATA, "|", manifest["bundle_of"])

## 1. The national peak snapshot (15 Jan 2026 12:15 UTC, 11,329 MW)
The bundle includes the pandapower net of this snapshot with its loads and dispatch. Re-solving it gives the stored result.

In [ ]:
import pandas as pd, numpy as np
net = pp.from_json(str(DATA / "hv/CORE-3787-REN_peak_20260115_1215_solved.json"))
opts = dict(algorithm="nr", init="dc", calculate_voltage_angles=True, max_iteration=50, tolerance_mva=1e-6, enforce_q_lims=True, numba=False)
pp.runpp(net, **opts)
print("converged:", net.converged)
print(f"load {net.res_load.p_mw.sum():,.0f} MW | line losses {net.res_line.pl_mw.sum():.1f} MW | max line loading {net.res_line.loading_percent.max():.1f}%")
stored = pd.read_parquet(DATA / "hv/peak_20260115_1215_line_loading.parquet").set_index("line_id").loading_percent
mine = pd.Series(net.res_line.loading_percent.values, index=net.line.line_id)
print("max |difference| to the stored result (percentage points):", float((mine - stored).abs().max()))

## 2. Line loading map

In [ ]:
import json, matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
lines = pd.read_parquet(DATA / "hv/lines.parquet").set_index("line_id")
geom = lines.geometry_json.map(json.loads)
ld = mine.reindex(lines.index)
bins, cols = [0, 20, 40, 60, 80, 100, 1e9], ["#4d9a4d", "#8bb04a", "#c9b43d", "#d98b35", "#d4552e", "#b0172a"]
fig, ax = plt.subplots(figsize=(5, 8))
for i in range(6):
    sel = (ld >= bins[i]) & (ld < bins[i + 1])
    ax.add_collection(LineCollection(list(geom[sel]), colors=cols[i], lw=0.5 + 0.4 * i, label=f"{bins[i]:.0f}–{bins[i+1]:.0f}%" if i < 5 else ">100%"))
ax.autoscale(); ax.set_aspect(1.3); ax.axis("off"); ax.legend(title="loading", loc="lower right", fontsize=8); plt.show()
top = ld.sort_values(ascending=False).head(5).to_frame("loading_percent").join(lines[["name", "voltage_kv", "length_km"]]); top

## 3. A month of loadings for the most loaded line
`hv/line_loading_2026-01.parquet` holds all 2,976 January snapshots (the release holds all 11 months).

In [ ]:
import duckdb
lid = top.index[0]
ts = duckdb.sql(f"select timestamp_utc, loading_percent from read_parquet('{DATA}/hv/line_loading_2026-01.parquet') where line_id = '{lid}' order by 1").df()
ax = ts.set_index("timestamp_utc").loading_percent.plot(figsize=(10, 3)); ax.axhline(100, ls="--", c="k", lw=0.8)
ax.set_ylabel("loading (%)"); ax.set_title(f"{lid} ({lines.at[lid, 'name']}), January 2026"); plt.show()
print(f"intervals above 100%: {(ts.loading_percent > 100).sum()} of {len(ts)}")

## 4. A simple N-1 check at the peak
Take each 400 kV line out in turn, re-solve, and record the worst loading elsewhere. This is an illustration, not an operator security assessment: switching states are inferred and generator Q limits are engineering proxies.

In [ ]:
import copy
res = []
cand = net.line.index[(net.line.voltage_kv == 400) & net.line.in_service]
for i in cand:
    n1 = copy.deepcopy(net); n1.line.at[i, "in_service"] = False
    try:
        pp.runpp(n1, **{**opts, "init": "results"})
        res.append((net.line.at[i, "line_id"], n1.res_line.loading_percent.max(), (n1.res_line.loading_percent > 100).sum()))
    except pp.LoadflowNotConverged:
        res.append((net.line.at[i, "line_id"], np.nan, np.nan))
n1 = pd.DataFrame(res, columns=["outaged_line", "max_loading_percent", "lines_over_100"]).sort_values("max_loading_percent", ascending=False)
print(f"{len(n1)} outages; not converged: {n1.max_loading_percent.isna().sum()}")
n1.head(10)

**Citation.** SimPT-Power is a simulation database built from public data. It is *not* the operator's network or a digital twin: converged power flows show numerical consistency, not agreement with operator state estimates. Please cite the archived release (Zenodo DOI) and the data paper. Data licences follow the sources (E-REDES CC BY 4.0, OpenStreetMap-derived layers ODbL); see `SOURCE_LICENSES.md` in the release.